# Notebook 18 (Revised) — PVOD frozen-specification replication

**Research stage:** cross-dataset *retrospective* replication, **not** a blind independent test. The historical PVOD study has previously inspected this release. The purpose is to assess whether the **SOLETE-selected feature/model specifications** retain utility when **independently refitted on each PVOD station**. No SOLETE fitted weights are transferred.

**Inputs:** the existing project `src/pvod.py`, `src/preprocessing.py`, `src/classical_models.py`, the committed SOLETE model-selection lock, and your local, untracked `external_data/PVOD/metadata.csv` and `station00.csv`–`station09.csv`.

**One notebook, three explicitly separate runs (set the environment variable before launching/restarting Jupyter):**

1. `PVOD_EVALUATION_STAGE=audit` (default): verifies the source/lock, re-runs the Notebook 17 population counts, **no fitting, scoring, or writing**.
2. `PVOD_EVALUATION_STAGE=fit`: trains **30** station–horizon LightGBM models; saves *local, Git-ignored* model checkpoints and writes **one fit-only manifest**, but **does not materialize test features, predict, or report PVOD test metrics**. **Commit the fit manifest and this notebook before scoring.**
3. `PVOD_EVALUATION_STAGE=score`: **only if the fit manifest is already committed**, verifies the frozen notebook, source/data/weight hashes and then scores the fixed test populations. No training is allowed in this stage.

Do not modify Notebook 17, historical PVOD notebooks, or past model archives. The stage is controlled **only by a process environment variable**, not by editing this notebook between phases. Run cells from the top with a fresh kernel in each stage. Before any fit, commit Notebook 17 and Notebook 18. **Do not interpret a successful local `audit` as a new model result.**

## Frozen selection and evaluation policy

| Horizon | Selected model | PVOD representation | Input width |
|---|---|---|---:|
| 15 min | `LightGBM_C1` | F1: F0 + target-valid Gregorian | 86 |
| 30 min | `LightGBM_C1` | F2: F1 + target-valid solar | 91 |
| 60 min | `LightGBM_C1` | F2 | 91 |

Model fixed at 300 trees, learning rate 0.03, 15 leaves, seed 42, with all remaining settings from the frozen `src.classical_models` LightGBM factory. No tuning, early stopping, scaling, clipping, sample substitution or feature reselection.

**Splits and availability:** ten station-specific 65/15/20 time-span splits; exact 15-minute gap-safe samples and target-valid-time split assignment from `src.pvod`. Refit only on train+validation labels with target-valid time strictly **earlier than the earliest test forecast issue across all three horizons in that station**. The 4 or more purged near-test rows are not used. Solar day is `solar_elevation_deg > 0` at the target-valid time.

**Primary comparison:** per-station MAE/RMSE in capacity-normalized p.u. against **issue-time PV persistence** on **all** eligible test rows; equal-station macro summaries, with separate pooled observation-weighted summaries. **Previous-day target-valid-time persistence** is evaluated on a separately declared, matched available subset only. All/daylight scopes are fixed before scoring. MW errors are station-specific supplemental metrics. Never clip targets or predictions.

**Limitations:** PVOD was previously studied; results are replication, not fresh untouched external testing. The weather fields are indexed at issue/past times, but real-world NWP publication timestamps are not verified. This analysis is not operational NWP availability validation or evidence of a causal Panchang effect.

In [1]:
from __future__ import annotations
from pathlib import Path
from collections import defaultdict
import gc
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import numpy as np
import pandas as pd
from IPython.display import display

STAGE = os.environ.get("PVOD_EVALUATION_STAGE", "audit").strip().lower()
assert STAGE in {"audit", "fit", "score"}, "Use audit, fit, or score only"

paths = []
if os.environ.get("SOLETE_PROJECT_ROOT"):
    paths.append(Path(os.environ["SOLETE_PROJECT_ROOT"]).expanduser())
paths.extend((Path.cwd(), *Path.cwd().parents))
PROJECT_ROOT = next((p.resolve() for p in paths if (p / ".git").exists()
    and (p / "src/pvod.py").is_file()
    and (p / "results/development_only/revised_model_selection_lock_v1.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run inside the project, or set SOLETE_PROJECT_ROOT")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NB_REL = Path("notebooks/18_revised_pvod_external_replication.ipynb")
NB_PATH = PROJECT_ROOT / NB_REL
PVOD_ROOT = PROJECT_ROOT / "external_data/PVOD"
LOCK_REL = Path("results/development_only/revised_model_selection_lock_v1.json")
FIT_REL = Path("results/external_validation/revised_pvod_lightgbm_transfer_v1_fit_manifest.json")
FIT_PATH = PROJECT_ROOT / FIT_REL
WEIGHTS_REL = Path("results/pvod_transfer_checkpoints_v1")
WEIGHTS_DIR = PROJECT_ROOT / WEIGHTS_REL
SCORE_DIR = PROJECT_ROOT / "results/external_validation"
SCORES_PATH = SCORE_DIR / "revised_pvod_lightgbm_transfer_v1_station_metrics.csv"
SUMMARY_PATH = SCORE_DIR / "revised_pvod_lightgbm_transfer_v1_aggregates.csv"
SCORE_MANIFEST_PATH = SCORE_DIR / "revised_pvod_lightgbm_transfer_v1_score_manifest.json"
ROW_ARCHIVE = WEIGHTS_DIR / "revised_pvod_lightgbm_transfer_v1_predictions.csv.gz"

if not PVOD_ROOT.is_dir():
    raise FileNotFoundError(f"PVOD local release missing: {PVOD_ROOT}")
if not NB_PATH.exists():
    raise FileNotFoundError(f"Place this notebook at {NB_PATH} before execution")
print("Stage:", STAGE.upper())
print("Repository:", PROJECT_ROOT)
print("PVOD directory (must stay outside Git):", PVOD_ROOT)
print("Fit manifest:", FIT_REL)
print("Weights directory (Git-ignored by repository rule):", WEIGHTS_REL)
print("STAGE=audit never fits, scores, or writes; STAGE=fit never scores; STAGE=score never fits")

Stage: FIT
Repository: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project
PVOD directory (must stay outside Git): /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/external_data/PVOD
Fit manifest: results/external_validation/revised_pvod_lightgbm_transfer_v1_fit_manifest.json
Weights directory (Git-ignored by repository rule): results/pvod_transfer_checkpoints_v1
STAGE=audit never fits, scores, or writes; STAGE=fit never scores; STAGE=score never fits


In [2]:
from src.classical_models import (
    LIGHTGBM_CONFIGS, LIGHTGBM_FIXED_PARAMS, build_lightgbm_model,
)
from src.pvod import (
    PVOD_CADENCE, PVOD_EXPECTED_DIMS, PVOD_HORIZONS,
    PVOD_LIGHTGBM_BASE_CONFIG, PVOD_TIMESTAMP_CORRECTIONS,
    PVOD_METEOROLOGICAL_COLUMNS, build_pvod_site_registry,
    clean_pvod_station, build_pvod_split_boundaries, build_pvod_f0,
    build_pvod_solar_features, build_pvod_sample_indices, materialize_pvod_design,
)
from src.preprocessing import build_gregorian_features

LOCK = json.loads((PROJECT_ROOT / LOCK_REL).read_text(encoding="utf-8"))
assert LOCK["evaluation_policy"]["terminal_access_authorized"] is False  # SOLETE lock: unchanged
SELECTED = {int(c["horizon_minutes"]): c for c in LOCK["selected_candidates"]}
assert set(SELECTED) == set(PVOD_HORIZONS) == {15, 30, 60}
FAMILIES = {h: SELECTED[h]["feature_family"] for h in (15, 30, 60)}
assert FAMILIES == {15: "F1", 30: "F2", 60: "F2"}
CONFIG = {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 15}
assert dict(LIGHTGBM_CONFIGS[0]) == CONFIG
assert all(c["model"] == "LightGBM" and c["candidate_id"] == "LightGBM_C1"
               and c["configuration"] == CONFIG for c in SELECTED.values())
assert {**LIGHTGBM_FIXED_PARAMS, **CONFIG} == PVOD_LIGHTGBM_BASE_CONFIG
assert PVOD_CADENCE == pd.Timedelta(minutes=15)
assert PVOD_EXPECTED_DIMS["F1"] == 86 and PVOD_EXPECTED_DIMS["F2"] == 91
assert set(PVOD_TIMESTAMP_CORRECTIONS) == {"station00"}

STATIONS = tuple(f"station{i:02d}" for i in range(10))
STATION_FILES = {s: PVOD_ROOT / f"{s}.csv" for s in STATIONS}
assert all(p.is_file() for p in [PVOD_ROOT / "metadata.csv", *STATION_FILES.values()])
assert {p.stem for p in PVOD_ROOT.glob("station*.csv")} == set(STATIONS)
REGISTRY = build_pvod_site_registry(pd.read_csv(PVOD_ROOT / "metadata.csv"))
assert len(REGISTRY) == 10 and set(REGISTRY["station"]) == set(STATIONS)
SITE_LOOKUP = REGISTRY.set_index("station").loc[list(STATIONS)]
assert SITE_LOOKUP["capacity_MW"].gt(0).all()
print("FROZEN MODEL SELECTION / PVOD METADATA: PASSED")
display(SITE_LOOKUP[["capacity_MW", "latitude_deg", "longitude_deg"]])

FROZEN MODEL SELECTION / PVOD METADATA: PASSED


,capacity_MW,latitude_deg,longitude_deg
station,,,
station00,6.6,38.047780,114.951390
station01,20.0,38.183060,117.457220
station02,17.0,38.057280,114.198870
station03,20.0,38.109560,114.114940
station04,20.0,39.515500,114.867670
station05,35.0,38.235500,114.123600
station06,15.0,36.898910,114.548410
station07,20.0,36.644030,113.641870
station08,20.0,36.707610,113.899990


## Immutable population reference from the executed Notebook 17

Counts below are **not new forecasts**. They are the 30 refit/test population counts in Notebook 17's saved output, used here as a fail-closed reconciliation guard. All 30 prior-day baseline gaps were zero in that preflight; recheck again in `score` stage. A changed PVOD file, source, split, or sample builder must cause failure rather than silently redefining the target population.

In [3]:
# (safe refit rows, gap-safe test rows, previous-day baseline unavailable rows)
EXPECTED = {
    "station00": {15:(22900,5798,0),30:(22898,5798,0),60:(22894,5798,0)},
    "station01": {15:(26626,6681,0),30:(26625,6681,0),60:(26623,6681,0)},
    "station02": {15:(24321,5719,0),30:(24319,5717,0),60:(24315,5713,0)},
    "station03": {15:(11650,2937,0),30:(11649,2937,0),60:(11647,2937,0)},
    "station04": {15:(26626,6681,0),30:(26625,6681,0),60:(26623,6681,0)},
    "station05": {15:(7656,1939,0),30:(7655,1939,0),60:(7653,1939,0)},
    "station06": {15:(24279,6239,0),30:(24274,6238,0),60:(24264,6236,0)},
    "station07": {15:(26049,6681,0),30:(26047,6681,0),60:(26043,6681,0)},
    "station08": {15:(26241,6681,0),30:(26239,6681,0),60:(26235,6681,0)},
    "station09": {15:(19270,4529,0),30:(19267,4527,0),60:(19261,4523,0)},
}
assert set(EXPECTED) == set(STATIONS) and all(set(v) == {15,30,60} for v in EXPECTED.values())

SOURCE_RELS = [Path("src/pvod.py"), Path("src/classical_models.py"),
               Path("src/preprocessing.py"), LOCK_REL]

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for buf in iter(lambda: f.read(1024 * 1024), b""):
            h.update(buf)
    return h.hexdigest()

def source_sha_from_document(raw: bytes | str) -> str:
    # Ignore volatile outputs, execution counts, cell IDs and metadata.
    data = json.loads(raw)
    canonical = [(c["cell_type"], "".join(c["source"])) for c in data["cells"]]
    return hashlib.sha256(json.dumps(canonical, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()

def notebook_source_sha() -> str:
    return source_sha_from_document(NB_PATH.read_bytes())

def run_git(*args: str) -> bytes:
    return subprocess.check_output(["git", "-C", str(PROJECT_ROOT), *args], stderr=subprocess.STDOUT)

def utcstr(t) -> str:
    return pd.Timestamp(t).isoformat()

def station_cutoff_and_masks(samples):
    first_issues = {}
    for h in (15,30,60):
        s = samples[h]
        assert {"train", "validation", "test"}.issubset(set(s["split"]))
        first_issues[h] = pd.Timestamp(s.loc[s["split"].eq("test"),"issue_time_utc"].iloc[0])
    cutoff = min(first_issues.values())
    masks = {h: samples[h]["split"].isin(["train", "validation"])
             & samples[h]["target_time_utc"].lt(cutoff) for h in (15,30,60)}
    for h in (15,30,60):
        assert masks[h].any()
        assert (samples[h].loc[masks[h],"target_time_utc"] < first_issues[h]).all()
        assert not (masks[h] & samples[h]["split"].eq("test")).any()
    return cutoff, first_issues, masks

def station_inputs(station):
    site = SITE_LOOKUP.loc[station]
    raw = pd.read_csv(STATION_FILES[station])
    required = {"date_time", "power", *PVOD_METEOROLOGICAL_COLUMNS}
    assert required.issubset(raw.columns)
    assert raw[list(PVOD_METEOROLOGICAL_COLUMNS)+["power"]].notna().all().all()
    clean = clean_pvod_station(raw, station_id=station, capacity_kw=float(site["capacity_kW"]))
    assert clean["timestamp_utc"].is_monotonic_increasing and clean["timestamp_utc"].is_unique
    assert np.isfinite(clean["power_pu"].to_numpy(dtype=float)).all()
    f0 = build_pvod_f0(clean)
    times = pd.DatetimeIndex(clean["timestamp_utc"])
    f1 = build_gregorian_features(times)
    f2 = build_pvod_solar_features(times, latitude_deg=float(site["latitude_deg"]),
                                   longitude_deg=float(site["longitude_deg"]))
    assert f0.shape == (len(clean),71) and f1.shape == (len(clean),15) and f2.shape == (len(clean),5)
    boundary = build_pvod_split_boundaries({station:clean}).iloc[0]
    samples = build_pvod_sample_indices(clean,f0,boundary,f2)
    cutoff,first_issues,safe = station_cutoff_and_masks(samples)
    return clean,times,f0,f1,f2,samples,cutoff,first_issues,safe

def materialize(samples,f0,f1,f2,times,family):
    return materialize_pvod_design(samples,f0=f0,f1=f1,f2=f2,
        f3=pd.DataFrame(index=times),f4=pd.DataFrame(index=times),feature_block=family)

def fingerprint_columns(columns):
    return hashlib.sha256(json.dumps(list(columns),ensure_ascii=False).encode()).hexdigest()

def record_for_population(station,h,s,cutoff,first_issues,safe,times):
    tr = s.loc[safe[h]]
    te = s.loc[s["split"].eq("test")]
    yday = pd.DatetimeIndex(te["target_time_utc"]) - pd.Timedelta(days=1)
    yesterday_unavailable = int((~yday.isin(times)).sum())
    observed = (len(tr),len(te),yesterday_unavailable)
    assert observed == EXPECTED[station][h], (station,h,observed,EXPECTED[station][h])
    assert ((s["target_time_utc"] - s["issue_time_utc"]) == pd.Timedelta(minutes=h)).all()
    assert (tr["target_time_utc"] < cutoff).all()
    assert (te["issue_time_utc"] >= first_issues[h]).all()
    assert s["target_time_utc"].is_unique and s["issue_time_utc"].is_unique
    purged = int(s["split"].isin(["train","validation"]).sum()-len(tr))
    assert purged == 4, (station,h,"changed purge size",purged)
    return {"station":station,"horizon_min":h,"family":FAMILIES[h],
        "refit_rows_safe":len(tr),"purged_near_test":purged,"test_rows_gap_safe":len(te),
        "test_rows_yesterday_unavailable":yesterday_unavailable,
        "refit_cutoff_utc":utcstr(cutoff),"first_test_issue_utc":utcstr(first_issues[h]),
        "first_test_valid_utc":utcstr(te["target_time_utc"].iloc[0])}

SRC_HASHES = {str(p):sha256_file(PROJECT_ROOT/p) for p in SOURCE_RELS}
DATA_HASHES = {"metadata.csv":sha256_file(PVOD_ROOT/"metadata.csv")}
DATA_HASHES.update({f"{s}.csv":sha256_file(STATION_FILES[s]) for s in STATIONS})
NOTEBOOK_SHA = notebook_source_sha()
print("Notebook source hash:",NOTEBOOK_SHA)
print("Source and external-data SHA-256 recorded in memory (never use external data in Git)")

Notebook source hash: 7fbdca13c12fe7578539d8d48a694b84799481c6e180da9af2f5c74158a114c2
Source and external-data SHA-256 recorded in memory (never use external data in Git)


## Freeze / commit gates

The `fit` phase refuses to overwrite any existing fit manifest, model weights, or score files. It checks that the notebook **source** is committed (execution-output changes are allowed), and frozen source/selection files have no tracked working-tree changes; start Jupyter after committing this notebook (and Notebook 17). Model weights are saved locally under the repository's existing ignore rule `results/pvod_*_checkpoints*/` and are **not committed**. The fit manifest stores their checksums and must be committed **as a separate checkpoint** before `score` can run.

In `score`, the committed fit manifest must byte-match the local manifest; every saved model, local PVOD CSV, source file, notebook source and model schema must match its recorded hash. A score attempt with missing/altered checkpoints fails. No test data are used to select models or stop fitting.

In [4]:
if STAGE == "fit":
    tracked = [str(NB_REL), *(str(p) for p in SOURCE_RELS)]
    run_git("ls-files", "--error-unmatch", *tracked)
    if run_git("status", "--porcelain", "--", *(str(p) for p in SOURCE_RELS)).strip():
        raise RuntimeError("Frozen source/lock has uncommitted tracked edits before FIT")
    # A previous audit run may have changed notebook OUTPUTS; that is safe.
    # Notebook SOURCE must match the already-committed source exactly.
    committed_nb_source = source_sha_from_document(run_git("show", f"HEAD:{NB_REL.as_posix()}"))
    if committed_nb_source != NOTEBOOK_SHA:
        raise RuntimeError("Notebook 18 SOURCE differs from Git HEAD: commit the source before FIT")
    if FIT_PATH.exists() or any(WEIGHTS_DIR.glob("*.txt")):
        raise FileExistsError("Previous fit manifest/model weights exist; do not overwrite them")
    if any(p.exists() for p in (SCORES_PATH,SUMMARY_PATH,SCORE_MANIFEST_PATH,ROW_ARCHIVE)):
        raise FileExistsError("Existing PVOD result files; do not overwrite")
    print("FIT GATE PASSED: tracked source clean, no historical/replication artifacts overwritten")
elif STAGE == "score":
    if not FIT_PATH.is_file():
        raise FileNotFoundError("Fit phase and commit the fit-only manifest before scoring")
    manifest_bytes = FIT_PATH.read_bytes()
    frozen_bytes = run_git("show", f"HEAD:{FIT_REL.as_posix()}")
    if manifest_bytes != frozen_bytes:
        raise RuntimeError("Fit manifest is not identical to the Git-committed copy at HEAD")
    FIT_MANIFEST = json.loads(manifest_bytes)
    assert FIT_MANIFEST["phase"] == "fit_only_no_test_scoring"
    assert FIT_MANIFEST["source_sha256"] == SRC_HASHES
    assert FIT_MANIFEST["dataset_sha256"] == DATA_HASHES
    assert FIT_MANIFEST["notebook_source_sha256"] == NOTEBOOK_SHA
    assert FIT_MANIFEST["feature_families"] == {str(h):FAMILIES[h] for h in (15,30,60)}
    assert FIT_MANIFEST["configuration"] == CONFIG
    assert len(FIT_MANIFEST["fits"]) == 30
    assert not any(p.exists() for p in (SCORES_PATH,SUMMARY_PATH,SCORE_MANIFEST_PATH,ROW_ARCHIVE)), (
        "Results exist already; never overwrite a previously computed external score")
    for fit in FIT_MANIFEST["fits"]:
        weight = PROJECT_ROOT / fit["model_path"]
        assert weight.is_file() and sha256_file(weight) == fit["model_sha256"], weight
    FIT_COMMIT = run_git("rev-parse", "HEAD").decode().strip()
    print("SCORE GATE PASSED: SHA-256 / committed-manifest checks PASSED")
    print("Fitted-model manifest committed at HEAD:",FIT_COMMIT)
else:
    print("AUDIT GATE: no writes, model fits, or predictions are authorized")

FIT GATE PASSED: tracked source clean, no historical/replication artifacts overwritten


## Stage A — independent sample audit, or 30 model fits (never test scores)

`audit` reconstructs the Notebook 17 counts and schema only. `fit` uses those identical eligible refit rows, trains one fresh LightGBM for each station × horizon, stores exactly 30 model text files and a JSON fit manifest containing reproducibility hashes. **Neither branch materializes test features or calculates test error.** The audit includes only test timestamp/count metadata; the existing sample builder may read test targets to determine eligibility, as disclosed in Notebook 17.

In [5]:
population_rows=[]
fit_rows=[]
schema_hashes=defaultdict(set)
if STAGE in {"audit","fit"}:
    if STAGE == "fit":
        WEIGHTS_DIR.mkdir(parents=True,exist_ok=True)
    for station in STATIONS:
        clean,times,f0,f1,f2,samples,cutoff,first_issues,safe = station_inputs(station)
        for h in (15,30,60):
            s=samples[h]
            pop=record_for_population(station,h,s,cutoff,first_issues,safe,times)
            family=FAMILIES[h]
            refit_s = s.loc[safe[h]].copy()
            design=materialize(refit_s,f0,f1,f2,times,family)
            X, y = design["X"],design["y_pu"]
            assert X.shape == (len(refit_s),PVOD_EXPECTED_DIMS[family])
            assert X.index.equals(y.index) and np.isfinite(X.to_numpy(dtype=float)).all()
            assert np.isfinite(y.to_numpy(dtype=float)).all()
            schema=fingerprint_columns(X.columns)
            schema_hashes[family].add(schema)
            pop["schema_sha256"] = schema
            population_rows.append(pop)
            if STAGE == "fit":
                model_path=WEIGHTS_DIR / f"{station}_{h:02d}min_{family}_LightGBM_C1.txt"
                if model_path.exists():
                    raise FileExistsError(model_path)
                model = build_lightgbm_model(CONFIG)
                model.fit(X, y)  # No eval_set, callbacks, early stopping or predictions.
                assert int(model.booster_.current_iteration()) == 300
                assert model.booster_.num_feature() == X.shape[1]
                model.booster_.save_model(str(model_path))
                assert list(model.booster_.feature_name()) == list(X.columns)
                fit_rows.append({**pop,"model_path":model_path.relative_to(PROJECT_ROOT).as_posix(),
                    "model_sha256":sha256_file(model_path),"feature_columns":int(X.shape[1]),
                    "tree_count":int(model.booster_.current_iteration())})
                del model
            del design,X,y,refit_s
        print(f"{station}: 3 horizon populations + schemas " + ("FITTED" if STAGE=="fit" else "AUDITED"))
        del clean,times,f0,f1,f2,samples
        gc.collect()
    assert len(population_rows)==30
    assert {k:len(v) for k,v in schema_hashes.items()}=={"F1":1,"F2":1}
    audit_table=pd.DataFrame(population_rows).sort_values(["station","horizon_min"]).reset_index(drop=True)
    display(audit_table[["station","horizon_min","family","refit_rows_safe","purged_near_test","test_rows_gap_safe"]])
    print("NOTEBOOK 17 POPULATION RECONCILIATION: PASSED (30/30)")

if STAGE=="fit":
    assert len(fit_rows)==30
    FIT_REPORT={"format_version":1,"phase":"fit_only_no_test_scoring",
        "study_classification":"cross_dataset_retrospective_replication_previously_studied_PVOD",
        "training_design":"station_specific_train_plus_validation_labels_before_earliest_test_issue",
        "no_test_predictions_or_metrics_created":True,
        "feature_families":{str(h):FAMILIES[h] for h in (15,30,60)},
        "candidate_id":"LightGBM_C1","configuration":CONFIG,
        "full_lightgbm_parameters":PVOD_LIGHTGBM_BASE_CONFIG,
        "source_sha256":SRC_HASHES,"dataset_sha256":DATA_HASHES,
        "notebook_source_sha256":NOTEBOOK_SHA,
        "python_version":sys.version.split()[0],
        "dependency_versions":{k:importlib.metadata.version(k)
             for k in ("lightgbm","numpy","pandas","pvlib","scikit-learn")},
        "fits":sorted(fit_rows,key=lambda x:(x["station"],x["horizon_min"]))}
    FIT_PATH.parent.mkdir(parents=True,exist_ok=True)
    with FIT_PATH.open("x",encoding="utf-8") as f:
        f.write(json.dumps(FIT_REPORT,indent=2,sort_keys=True,ensure_ascii=False)+"\n")
    print("30 FITS FROZEN LOCALLY; ZERO TEST SCORES")
    print("Commit this tracked FIT-ONLY manifest before setting PVOD_EVALUATION_STAGE=score:",FIT_REL)
else:
    print("No fits or fit manifest created in this stage")

station00: 3 horizon populations + schemas FITTED
station01: 3 horizon populations + schemas FITTED
station02: 3 horizon populations + schemas FITTED
station03: 3 horizon populations + schemas FITTED
station04: 3 horizon populations + schemas FITTED
station05: 3 horizon populations + schemas FITTED
station06: 3 horizon populations + schemas FITTED
station07: 3 horizon populations + schemas FITTED
station08: 3 horizon populations + schemas FITTED
station09: 3 horizon populations + schemas FITTED


,station,horizon_min,family,refit_rows_safe,purged_near_test,test_rows_gap_safe
0,station00,15,F1,22900,4,5798
1,station00,30,F2,22898,4,5798
2,station00,60,F2,22894,4,5798
3,station01,15,F1,26626,4,6681
4,station01,30,F2,26625,4,6681
5,station01,60,F2,26623,4,6681
6,station02,15,F1,24321,4,5719
7,station02,30,F2,24319,4,5717
8,station02,60,F2,24315,4,5713
9,station03,15,F1,11650,4,2937


NOTEBOOK 17 POPULATION RECONCILIATION: PASSED (30/30)
30 FITS FROZEN LOCALLY; ZERO TEST SCORES
Commit this tracked FIT-ONLY manifest before setting PVOD_EVALUATION_STAGE=score: results/external_validation/revised_pvod_lightgbm_transfer_v1_fit_manifest.json


## Stage B — score strictly after the committed fit checkpoint

This cell does nothing unless `PVOD_EVALUATION_STAGE=score`. It re-derives the exact same sample populations, verifies each fit record against the Git-frozen manifest, loads saved LightGBM **weights** (without fitting), and produces strictly fixed comparisons. It uses issue-time persistence for the complete gap-safe test population, and prior-day **target-valid** persistence only where that time exists. There is **no prediction clipping**.

Metrics are computed for `all` and `daylight` at two sample scopes: `full_gap_safe` (model and issue persistence) and `yesterday_matched` (same rows for all three methods). Each row records its own sample count. Predictions are saved only in the local Git-ignored checkpoint directory for audit. Station metrics and macro/pooled summaries are separate **new** tracked CSVs.

In [6]:
def error_stats(y_true,pred):
    a=np.asarray(y_true,dtype=np.float64)
    b=np.asarray(pred,dtype=np.float64)
    assert a.ndim==b.ndim==1 and len(a)==len(b) and len(a)>0
    assert np.isfinite(a).all() and np.isfinite(b).all()
    e=a-b
    return float(np.mean(np.abs(e))),float(np.sqrt(np.mean(e**2)))

def evaluate_station_rows(station,h,capacity_mw,df):
    results=[]
    for scope,daymask in [("all",np.ones(len(df),dtype=bool)),
                          ("daylight",df["daylight"].to_numpy(dtype=bool))]:
        for population,mask in [("full_gap_safe",daymask),
                                ("yesterday_matched",daymask & df["yesterday_available"].to_numpy(dtype=bool))]:
            if not mask.any():
                # Preflight showed prior-day availability at all 30 station/horizon populations;
                # fail rather than silently excluding any station from equal-site macro metrics.
                raise AssertionError((station,h,scope,population,"empty comparison subset"))
            y=df.loc[mask,"observed_pu"].to_numpy(dtype=float)
            methods=[("LightGBM_C1", "prediction_pu"),
                     ("issue_persistence","issue_persistence_pu")]
            if population=="yesterday_matched":
                methods.append(("yesterday_persistence","yesterday_persistence_pu"))
            for name,col in methods:
                mae,rmse=error_stats(y,df.loc[mask,col].to_numpy(dtype=float))
                results.append({"station":station,"horizon_min":h,"feature_family":FAMILIES[h],
                    "scope":scope,"population":population,"method":name,
                    "n":int(mask.sum()),"capacity_MW":float(capacity_mw),
                    "mae_pu":mae,"rmse_pu":rmse,
                    "mae_MW":mae*capacity_mw,"rmse_MW":rmse*capacity_mw})
    return results

def aggregate_site_scores(metrics):
    rows=[]
    keys=["horizon_min","scope","population","method"]
    for (h,scope,population,method),g in metrics.groupby(keys,sort=True):
        assert len(g)==10 and g["station"].nunique()==10
        weights=g["n"].to_numpy(dtype=float)
        rows.append({"horizon_min":h,"scope":scope,"population":population,
            "method":method,"station_count":10,"pooled_test_rows":int(weights.sum()),
            "macro_mae_pu":float(g["mae_pu"].mean()),
            "macro_rmse_pu":float(g["rmse_pu"].mean()),
            "pooled_mae_pu":float(np.average(g["mae_pu"],weights=weights)),
            "pooled_rmse_pu":float(np.sqrt(np.average(g["rmse_pu"]**2,weights=weights)))})
    return pd.DataFrame(rows).sort_values(keys).reset_index(drop=True)

if STAGE == "score":
    from lightgbm import Booster
    fit_index={(r["station"],int(r["horizon_min"])):r for r in FIT_MANIFEST["fits"]}
    assert len(fit_index)==30
    metric_rows=[]
    pred_frames=[]
    for station in STATIONS:
        clean,times,f0,f1,f2,samples,cutoff,first_issues,safe=station_inputs(station)
        capacity=float(SITE_LOOKUP.loc[station,"capacity_MW"])
        observed_source=clean.set_index("timestamp_utc")["power_pu"]
        for h in (15,30,60):
            s=samples[h]
            pop=record_for_population(station,h,s,cutoff,first_issues,safe,times)
            fit=fit_index[(station,h)]
            for key,value in pop.items():
                assert fit[key]==value,(station,h,key,fit[key],value)
            # Recheck training schema without fitting or accessing any training predictions.
            refit_design=materialize(s.loc[safe[h]],f0,f1,f2,times,FAMILIES[h])
            assert fingerprint_columns(refit_design["X"].columns)==fit["schema_sha256"]
            del refit_design
            test_s=s.loc[s["split"].eq("test")].copy()
            design=materialize(test_s,f0,f1,f2,times,FAMILIES[h])
            X=design["X"]
            assert len(X)==fit["test_rows_gap_safe"] and X.shape[1]==fit["feature_columns"]
            assert fingerprint_columns(X.columns)==fit["schema_sha256"]
            weights=PROJECT_ROOT/fit["model_path"]
            booster=Booster(model_file=str(weights))
            assert booster.num_feature()==X.shape[1] and booster.current_iteration()==300
            assert booster.feature_name()==list(X.columns)
            pred=np.asarray(booster.predict(X,num_threads=1),dtype=np.float64)
            actual=design["y_pu"].to_numpy(dtype=np.float64)
            issues=pd.DatetimeIndex(test_s["issue_time_utc"])
            targets=pd.DatetimeIndex(test_s["target_time_utc"])
            issue_pred=observed_source.reindex(issues).to_numpy(dtype=np.float64)
            yesterday_pred=observed_source.reindex(targets-pd.Timedelta(days=1)).to_numpy(dtype=np.float64)
            day=test_s["target_solar_daylight"].to_numpy(dtype=int)
            assert set(np.unique(day)).issubset({0,1})
            assert day.sum()>0 and (day==0).sum()>0
            assert np.isfinite(actual).all() and np.isfinite(pred).all() and np.isfinite(issue_pred).all()
            assert int((~np.isfinite(yesterday_pred)).sum())==fit["test_rows_yesterday_unavailable"]
            outcome=pd.DataFrame({"station":station,"horizon_min":h,
                "issue_time_utc":issues.astype(str),"target_time_utc":targets.astype(str),
                "daylight":day,"observed_pu":actual,"prediction_pu":pred,
                "issue_persistence_pu":issue_pred,"yesterday_persistence_pu":yesterday_pred,
                "yesterday_available":np.isfinite(yesterday_pred)})
            metric_rows.extend(evaluate_station_rows(station,h,capacity,outcome))
            pred_frames.append(outcome)
            del booster,design,X,pred,actual,issue_pred,yesterday_pred,outcome
        print(f"{station}: three saved model weights verified and scored")
        del clean,times,f0,f1,f2,samples,observed_source
        gc.collect()
    station_metrics=pd.DataFrame(metric_rows).sort_values([
        "horizon_min","station","scope","population","method"]).reset_index(drop=True)
    aggregate_metrics=aggregate_site_scores(station_metrics)
    assert len(station_metrics)==30*(2*2+2*3)==300
    assert len(aggregate_metrics)==3*(2*2+2*3)==30
    assert station_metrics.groupby(["horizon_min","scope","population","method"])["station"].nunique().eq(10).all()
    display(station_metrics.loc[(station_metrics["scope"]=="all")
        & (station_metrics["population"]=="full_gap_safe"),[
        "station","horizon_min","method","n","mae_pu","rmse_pu"]])
    display(aggregate_metrics)
else:
    print("SCORING DISABLED in",STAGE.upper(),"stage")

SCORING DISABLED in FIT stage


## Stage C — write one-time score artifacts and show primary skill comparison

This cell is gated to `score`. No results are overwritten. It generates the full station table, macro/pooled table, a hash-bearing score manifest, and locally retained timestamp-level predictions. **The fitting checkpoint has already been committed at HEAD before this stage.** Score summaries are descriptive; bootstrap uncertainty and any further analyses require separate predeclared protocols, not ad hoc post-result tuning.

In [7]:
if STAGE=="score":
    # Primary skill = 1 - model MAE / issue-persistence MAE on identical full-population rows.
    primary=station_metrics.loc[(station_metrics["population"]=="full_gap_safe") &
        (station_metrics["scope"]=="all")].pivot(index=["station","horizon_min"],
             columns="method",values="mae_pu")
    assert primary.shape==(30,2)
    primary["mae_skill_vs_issue_persistence"]=(
        1-primary["LightGBM_C1"]/primary["issue_persistence"])
    print("Primary station comparisons (positive skill = lower MAE than issue persistence):")
    display(primary.reset_index())

    # Perform a final fail-closed check before the first new score artifact is written.
    assert FIT_PATH.read_bytes()==run_git("show",f"HEAD:{FIT_REL.as_posix()}")
    assert not any(p.exists() for p in (SCORES_PATH,SUMMARY_PATH,SCORE_MANIFEST_PATH,ROW_ARCHIVE))
    SCORE_DIR.mkdir(parents=True,exist_ok=True)
    station_metrics.to_csv(SCORES_PATH,index=False,mode="x",float_format="%.12g")
    aggregate_metrics.to_csv(SUMMARY_PATH,index=False,mode="x",float_format="%.12g")
    # raw per-timestamp predictions are intentionally local and ignored by Git
    pd.concat(pred_frames,ignore_index=True).to_csv(ROW_ARCHIVE,index=False,compression="gzip",float_format="%.12g")
    score_manifest={"format_version":1,"phase":"score_after_committed_fit",
        "study_classification":"cross_dataset_retrospective_replication_previously_studied_PVOD",
        "fit_manifest_sha256":sha256_file(FIT_PATH),"fit_commit":FIT_COMMIT,
        "no_refits_during_scoring":True,
        "metric_definition":"station MAE/RMSE in p.u.; equal-station macro and observation-pooled",
        "scopes":["all","daylight"],
        "populations":["full_gap_safe","yesterday_matched"],
        "primary_baseline":"issue-time PV persistence on full gap-safe test rows",
        "model_prediction_clipping":False,
        "station_metric_sha256":sha256_file(SCORES_PATH),
        "aggregate_metric_sha256":sha256_file(SUMMARY_PATH),
        "local_predictions_sha256":sha256_file(ROW_ARCHIVE),
        "total_station_horizon_models":30,
        "total_unique_horizon_station_test_rows":sum(EXPECTED[s][h][1] for s in STATIONS for h in (15,30,60))}
    with SCORE_MANIFEST_PATH.open("x",encoding="utf-8") as f:
        f.write(json.dumps(score_manifest,indent=2,sort_keys=True)+"\n")
    print("PVOD 30-MODEL SCORING COMPLETE; no configurations changed")
    print("Tracked score outputs:",SCORES_PATH.relative_to(PROJECT_ROOT),
          SUMMARY_PATH.relative_to(PROJECT_ROOT),SCORE_MANIFEST_PATH.relative_to(PROJECT_ROOT))
    print("Local-only timestamp predictions:",ROW_ARCHIVE.relative_to(PROJECT_ROOT))
else:
    print("NO SCORE ARTIFACTS WRITTEN (stage",STAGE.upper()+")")

NO SCORE ARTIFACTS WRITTEN (stage FIT)


## Stop point and integrity checklist

- `audit`: if population reconciliation passes, inspect the output; no fit/scoring/outputs were made. Commit Notebook 17 and Notebook 18, then run `fit` in a restarted Jupyter kernel.
- `fit`: ensure **30** saved model `.txt` files plus the fit-only JSON manifest; commit **only** the JSON fit manifest (along with any notebook source changes made before fitting), **not** checkpoint weights or the local PVOD dataset. Note and preserve the Git commit SHA. Do not score before this checkpoint is committed.
- `score`: requires identical local weights/data/notebook/source and the already-committed manifest. Confirm all 30 stations × horizons, both scopes, the full and matched populations, per-station and aggregate p.u. metrics. Commit new score artifacts after scientific review.

**Do not clean historical notebooks yet.** We should review the scored external replication and limitations before the final manuscript update and notebook cleanup.

**Note:** the fit manifest is an audit checkpoint; it cannot prove historical PVOD blindness. Keep the existing PVOD prior-study exposure explicitly disclosed.